# SOLID Principles in Programming

SOLID is a set of five design principles used to create robust, testable, extensible, and maintainable object-oriented software.

## Meaning of SOLID

- **S** - Single Responsibility Principle (SRP)
- **O** - Open-Closed Principle (OCP)
- **L** - Liskov Substitution Principle (LSP)
- **I** - Interface Segregation Principle (ISP)
- **D** - Dependency Inversion Principle (DIP)

These principles were introduced by Robert C. Martin and are mainly applied to object-oriented programming. They help reduce tight coupling and make software easier to maintain, test, reuse, and extend.

## Importance

When classes depend heavily on one another, a change in one class can break unrelated behavior. Loose coupling reduces this risk by keeping responsibilities focused and dependencies small.

# 1. Single Responsibility Principle (SRP)

A class should have only one reason to change. In other words, one class should focus on one clear responsibility.

A class that manages order items, calculates totals, and processes payments mixes unrelated responsibilities. That makes it harder to maintain and test.

In [1]:
class Order:
    def __init__(self):
        self.items = []
        self.quantities = []
        self.prices = []

    def add_item(self, name, quantity, price):
        self.items.append(name)
        self.quantities.append(quantity)
        self.prices.append(price)

    def calculate_total(self):
        return sum(
            quantity * price
            for quantity, price in zip(self.quantities, self.prices)
        )

    def process_payment(self, payment_type, amount):
        total = self.calculate_total()
        if amount < total:
            print("Payment failed: not enough money.")
        else:
            print(f"Processing {payment_type} payment of {amount} units")
            print("Payment successful!")

print("=== Without SRP ===")
order = Order()
order.add_item("Book", 2, 300)
order.add_item("Pen", 5, 20)
print("Total:", order.calculate_total())
order.process_payment("Credit Card", 700)
order.process_payment("Credit Card", 500)

=== Without SRP ===
Total: 700
Processing Credit Card payment of 700 units
Payment successful!
Payment failed: not enough money.


## Example with SRP

Separate the responsibilities into focused classes:

- `Order` manages items and calculates the total.
- `PaymentProcessor` handles payment processing.

This makes each class easier to understand, test, and change.

In [2]:
class Order:
    def __init__(self):
        self.items = []
        self.quantities = []
        self.prices = []

    def add_item(self, name, quantity, price):
        self.items.append(name)
        self.quantities.append(quantity)
        self.prices.append(price)

    def calculate_total(self):
        return sum(
            quantity * price
            for quantity, price in zip(self.quantities, self.prices)
        )


class PaymentProcessor:
    def process_payment(self, order, payment_type, amount):
        total = order.calculate_total()
        if amount < total:
            print("Payment failed: not enough money.")
        else:
            print(f"Processing {payment_type} payment of {amount} units")
            print("Payment successful!")

print("=== With SRP ===")
order = Order()
order.add_item("Book", 2, 300)
order.add_item("Pen", 5, 20)
print("Total:", order.calculate_total())

payment_processor = PaymentProcessor()
payment_processor.process_payment(order, "Credit Card", 700)
payment_processor.process_payment(order, "Credit Card", 500)

=== With SRP ===
Total: 700
Processing Credit Card payment of 700 units
Payment successful!
Payment failed: not enough money.


## 2. Open-Closed Principle (OCP)

Software entities should be **open for extension but closed for modification**.

New behavior should be added without repeatedly changing stable, tested code. Abstraction and polymorphism help us extend a system safely.

In [3]:
class PaymentProcessorWithoutOCP:
    def process_payment(self, method, amount):
        if method == "credit_card":
            print(f"Processing credit card payment of {amount} units")
        elif method == "paypal":
            print(f"Processing PayPal payment of {amount} units")
        elif method == "crypto":
            print(f"Processing cryptocurrency payment of {amount} units")
        else:
            raise ValueError("Unsupported payment method")

processor = PaymentProcessorWithoutOCP()
processor.process_payment("credit_card", 100.0)
processor.process_payment("paypal", 250.0)

Processing credit card payment of 100.0 units
Processing PayPal payment of 250.0 units


In [4]:
from abc import ABC, abstractmethod


class PaymentMethod(ABC):
    @abstractmethod
    def pay(self, amount):
        pass


class CreditCardPayment(PaymentMethod):
    def pay(self, amount):
        print(f"Processing credit card payment of {amount} units")
        print("Payment successful!")


class PayPalPayment(PaymentMethod):
    def pay(self, amount):
        print(f"Processing PayPal payment of {amount} units")
        print("Payment successful!")


class BitcoinPayment(PaymentMethod):
    def pay(self, amount):
        print(f"Processing Bitcoin payment of {amount} units")
        print("Payment successful!")


class PaymentProcessor:
    def __init__(self, payment_method):
        self.payment_method = payment_method

    def process(self, amount):
        self.payment_method.pay(amount)


PaymentProcessor(CreditCardPayment()).process(100.0)
PaymentProcessor(PayPalPayment()).process(250.0)
PaymentProcessor(BitcoinPayment()).process(500.0)

Processing credit card payment of 100.0 units
Payment successful!
Processing PayPal payment of 250.0 units
Payment successful!
Processing Bitcoin payment of 500.0 units
Payment successful!


## OCP with notifications

The same idea applies to notification channels. A class with `if` statements for every channel must be modified whenever a new channel is added. A notifier abstraction lets us add new channels through new subclasses.

In [5]:
from abc import ABC, abstractmethod


class Notifier(ABC):
    @abstractmethod
    def send(self, message):
        pass


class EmailNotifier(Notifier):
    def send(self, message):
        print(f"Sending Email: {message}")


class SMSNotifier(Notifier):
    def send(self, message):
        print(f"Sending SMS: {message}")


class FacebookNotifier(Notifier):
    def send(self, message):
        print(f"Sending Facebook Message: {message}")


class Notification:
    def __init__(self, notifier):
        self.notifier = notifier

    def send(self, message):
        self.notifier.send(message)


Notification(EmailNotifier()).send("Your order has been shipped via Email!")
Notification(SMSNotifier()).send("Your order has been shipped via SMS!")
Notification(FacebookNotifier()).send("Your order has been shipped via Facebook Message!")

Sending Email: Your order has been shipped via Email!
Sending SMS: Your order has been shipped via SMS!
Sending Facebook Message: Your order has been shipped via Facebook Message!


# 3. Liskov Substitution Principle (LSP)

Subclasses should be replaceable with objects of their parent type without breaking the program.

A subclass must preserve the expectations and behavior established by its parent. If a type does not truly belong in a hierarchy, forcing it to inherit from that parent creates fragile code.

In [6]:
from abc import ABC, abstractmethod


class Coffee(ABC):
    @abstractmethod
    def serve(self):
        pass


class Espresso(Coffee):
    def serve(self):
        print("Serving a strong Espresso")


class Cappuccino(Coffee):
    def serve(self):
        print("Serving a creamy Cappuccino")


class Tea(Coffee):
    # Tea is not a kind of coffee, so this hierarchy is misleading.
    def serve(self):
        print("Serving Tea")


def order_coffee(beverage):
    beverage.serve()


order_coffee(Espresso())
order_coffee(Cappuccino())
order_coffee(Tea())  # The code runs, but the hierarchy violates LSP.

Serving a strong Espresso
Serving a creamy Cappuccino
Serving Tea


In [7]:
from abc import ABC, abstractmethod


class Beverage(ABC):
    @abstractmethod
    def serve(self):
        pass


class Coffee(Beverage):
    pass


class Espresso(Coffee):
    def serve(self):
        print("Serving a strong Espresso")


class Cappuccino(Coffee):
    def serve(self):
        print("Serving a creamy Cappuccino")


class Tea(Beverage):
    pass


class GreenTea(Tea):
    def serve(self):
        print("Serving a refreshing Green Tea")


class BlackTea(Tea):
    def serve(self):
        print("Serving a bold Black Tea")


def order_beverage(beverage):
    beverage.serve()


beverages = [Espresso(), Cappuccino(), GreenTea(), BlackTea()]
for beverage in beverages:
    order_beverage(beverage)

Serving a strong Espresso
Serving a creamy Cappuccino
Serving a refreshing Green Tea
Serving a bold Black Tea


# Summary

| Principle | Main idea |
|---|---|
| **SRP** | One class should have one responsibility. |
| **OCP** | Extend behavior without modifying stable code. |
| **LSP** | Subtypes must be safely substitutable for their parent type. |
| **ISP** | Clients should not depend on methods they do not use. |
| **DIP** | High-level and low-level modules should depend on abstractions. |

Applying SOLID generally leads to code that is more modular, maintainable, reusable, testable, and safer to extend. The principles are guidelines rather than rigid rules; apply them when they reduce coupling and clarify responsibilities.

## Conclusion

The **Liskov Substitution Principle (LSP)** ensures that subclasses can safely replace their parent classes without causing unexpected behavior.

In the compliant design, both coffee and tea types implement the shared `Beverage` abstraction. Client code can therefore work with any beverage without depending on its specific type.

This improves modularity, flexibility, maintainability, and extensibility while reducing the risk of unexpected behavior.